# cdfi-stress-tester — CDFI Portfolio Stress Testing Engine
## Demo: Monte Carlo Stress Testing for CDFI Loan Portfolios

This notebook demonstrates how to use cdfi-stress-tester to:
- Generate realistic CDFI loan portfolios for testing
- Apply standard stress scenarios (2008 recession, COVID, rate spike)
- Run Monte Carlo simulations with correlated NOI/rate/property shocks
- Compute Value at Risk (VaR) at 95% and 99% confidence levels
- Analyze capital adequacy under stress
- Compare scenarios side-by-side

Background: Every CDFI runs portfolio stress tests for IC reporting,
board governance, and CDFI Fund compliance. The math is the same everywhere
but each CDFI builds it from scratch in Excel. This library standardizes it.


In [1]:
import sys
sys.path.insert(0, '..')

from cdfistress import (
    generate_sample_portfolio,
    from_standard,
    create_recession_scenario, create_rate_shock_scenario,
    apply_shock_to_loan,
    MonteCarloEngine,
    build_correlation_matrix, default_correlations,
    value_at_risk, conditional_var, expected_loss, tail_loss,
    capital_adequacy, capital_adequacy_report, tier1_under_stress,
    generate_stress_report, scenario_comparison_table,
    STANDARD_SCENARIOS, SECTOR_DEFAULT_RATES,
)
import numpy as np

print("cdfi-stress-tester loaded successfully")
print(f"\nStandard scenarios: {list(STANDARD_SCENARIOS.keys())}")
print(f"Sectors tracked:    {list(SECTOR_DEFAULT_RATES.keys())}")

cdfi-stress-tester loaded successfully

Standard scenarios: ['2008_recession', 'covid_shock', 'rate_spike', 'regional_cre_crash', 'mild_downturn']
Sectors tracked:    ['retail', 'office', 'multifamily', 'industrial', 'healthcare', 'mixed_use', 'other']


## 1. Generate a Sample CDFI Portfolio

A reproducible 50-loan synthetic portfolio spanning the sectors the library
tracks: multifamily, retail, office, industrial, healthcare, and mixed use.

In [2]:
loans = generate_sample_portfolio(n=50, seed=42)

print(f"Generated {len(loans)} loans")
total_balance = sum(l.outstanding_balance for l in loans)
total_value = sum(l.property_value for l in loans)
print(f"Total Outstanding:      ${total_balance/1e6:.2f}MM")
print(f"Total Collateral Value: ${total_value/1e6:.2f}MM")

# Show first few loans
print("\nSample loans:")
print("=" * 75)
for loan in loans[:5]:
    print(
        f"  {loan.loan_id:<10} {loan.sector:<14} "
        f"${loan.outstanding_balance/1e6:>5.2f}MM  "
        f"DSCR: {loan.dscr:.2f}x  LTV: {loan.ltv*100:.0f}%"
    )

Generated 50 loans
Total Outstanding:      $205.04MM
Total Collateral Value: $273.09MM

Sample loans:
  CDFI-0001  mixed_use      $ 0.69MM  DSCR: 1.03x  LTV: 70%
  CDFI-0002  mixed_use      $ 1.15MM  DSCR: 1.08x  LTV: 73%
  CDFI-0003  healthcare     $ 0.70MM  DSCR: 1.33x  LTV: 69%
  CDFI-0004  retail         $ 4.92MM  DSCR: 0.79x  LTV: 81%
  CDFI-0005  mixed_use      $ 3.05MM  DSCR: 1.63x  LTV: 68%


## 2. Standard Stress Scenarios

The library ships with pre-calibrated scenarios based on real historical events.


In [3]:
print("Standard Stress Scenarios:")
print("=" * 80)
for key in STANDARD_SCENARIOS:
    s = from_standard(key)
    print(f"\n  {s.name} [{s.severity}]")
    print(f"    NOI shock:           {s.noi_shock*100:>6.1f}%")
    print(f"    Rate shock:          {s.rate_shock*100:>6.1f}%")
    print(f"    Property value:      {s.property_value_shock*100:>6.1f}%")
    print(f"    Default multiplier:  {s.default_rate_multiplier:>6.2f}x")


Standard Stress Scenarios:

  2008-Style Recession [severe]
    NOI shock:            -35.0%
    Rate shock:             0.0%
    Property value:       -40.0%
    Default multiplier:    4.00x

  COVID-Style Demand Shock [moderate]
    NOI shock:            -25.0%
    Rate shock:            -1.0%
    Property value:       -15.0%
    Default multiplier:    2.50x

  Rate Spike (+300 bps) [moderate]
    NOI shock:             -5.0%
    Rate shock:             3.0%
    Property value:       -20.0%
    Default multiplier:    1.80x

  Regional CRE Crash [severe]
    NOI shock:            -20.0%
    Rate shock:             1.0%
    Property value:       -50.0%
    Default multiplier:    3.00x

  Mild Cyclical Downturn [mild]
    NOI shock:            -10.0%
    Rate shock:             0.5%
    Property value:        -8.0%
    Default multiplier:    1.40x


## 3. Build the Monte Carlo Engine

The engine generates correlated shocks to NOI, interest rates, and property
values using multivariate normal distributions.


In [4]:
engine = MonteCarloEngine(loans=loans, available_capital=5_000_000)

print(f"Engine initialized")
print(f"Loans:              {len(engine.loans)}")
print(f"Available capital:  ${engine.available_capital:,.0f}")
print(f"\nDefault correlations (NOI, Rate, PropertyValue):")
print(default_correlations())


Engine initialized
Loans:              50
Available capital:  $5,000,000

Default correlations (NOI, Rate, PropertyValue):
[[ 1.  -0.3  0.7]
 [-0.3  1.  -0.6]
 [ 0.7 -0.6  1. ]]


## 4. Run a 2008-Style Recession Scenario

1,000 Monte Carlo iterations applying severe recession shocks to the portfolio.


In [5]:
scenario = from_standard("2008_recession")
result = engine.run_simulation(scenario, n_iterations=1000, seed=42)

print(f"Scenario: {result.scenario.name}")
print(f"Severity: {result.scenario.severity.upper()}")
print("=" * 50)
print(f"Expected Loss:        ${result.expected_loss:,.0f}")
print(f"VaR (95%):            ${result.var_95:,.0f}")
print(f"VaR (99%):            ${result.var_99:,.0f}")
print(f"Capital Adequacy:     {result.capital_adequacy_ratio:.2f}x")
print(f"Buffer Breaches:      {result.num_breaches} of 1000 simulations")


Scenario: 2008-Style Recession
Severity: SEVERE
Expected Loss:        $8,843,297
VaR (95%):            $15,033,139
VaR (99%):            $17,396,438
Capital Adequacy:     0.57x
Buffer Breaches:      858 of 1000 simulations


## 5. Generate Full Stress Report

In [6]:
report = generate_stress_report(result)
print(report)


CDFI PORTFOLIO STRESS TEST REPORT
Scenario: 2008-Style Recession  [SEVERE]

SCENARIO PARAMETERS
  NOI shock            : -35.0%
  Rate shock           : +0 bps
  Property value shock : -40.0%
  Default mult         : 4.0x baseline

LOSS METRICS
  Expected Loss        : $      8,843,297
  VaR (95%)            : $     15,033,139
  VaR (99%)            : $     17,396,438

CAPITAL ADEQUACY
  CAR vs Expected Loss : 0.57x
  Simulation breaches  : 858 / 1000 (85.8%)

STATUS: INSUFFICIENT


## 6. Compare Multiple Scenarios

Run the portfolio through each standard scenario to compare outcomes.


In [7]:
scenarios = [from_standard(k) for k in STANDARD_SCENARIOS.keys()]
results = [engine.run_simulation(s, n_iterations=500, seed=0) for s in scenarios]

# scenario_comparison_table returns a preformatted str, not a DataFrame
print("Scenario Comparison Table:")
print(scenario_comparison_table(results))

Scenario Comparison Table:
Scenario                       Severity               EL          VaR95          VaR99     CAR   Breaches
---------------------------------------------------------------------------------------------------------
2008-Style Recession           severe     $    8,769,697 $   15,572,653 $   19,507,034   0.57x     429/500
COVID-Style Demand Shock       moderate   $    5,193,687 $   10,176,538 $   12,372,394   0.96x     254/500
Rate Spike (+300 bps)          moderate   $    3,122,681 $    7,171,131 $    8,733,905   1.60x      94/500
Regional CRE Crash             severe     $    6,800,075 $   13,389,838 $   17,171,671   0.74x     343/500
Mild Cyclical Downturn         mild       $    2,476,091 $    6,029,387 $    7,761,442   2.02x      52/500


## 7. Custom Scenarios

Build your own scenarios — e.g. CRE-specific stress or rate spike only.


In [8]:
# Custom rate spike scenario.  property_value_shock is derived from the rate
# shock when it is not supplied.
custom = create_rate_shock_scenario(rate_shock=0.05)

result_custom = engine.run_simulation(custom, n_iterations=500, seed=0)
print(f"Custom Scenario: {custom.name}")
print(f"Expected Loss:    ${result_custom.expected_loss:,.0f}")
print(f"VaR (95%):        ${result_custom.var_95:,.0f}")

Custom Scenario: Rate Shock (+5 bps)
Expected Loss:    $3,524,333
VaR (95%):        $7,545,946


## 8. Custom Labelled Scenarios

`create_recession_scenario` takes `name` and `severity` labels so you can build
your own scenario without a second constructor.

Note: shocks are applied to **every** loan in the portfolio. A scenario's
`noi_shock`, `rate_shock`, `property_value_shock` and `default_rate_multiplier`
are scalars with no per-segment override, and `run_simulation` applies them to
every loan — so a label like "Retail Crash" would be misleading, and 0.1.0's
`create_sector_specific_scenario` (whose `sector` argument only built a display
string) was removed in 0.2.0 rather than dressed up with invented per-sector
calibrations.

**What is missing is calibration, not mechanism.** The engine already resolves
each loan's sector into a per-loan baseline default rate via
`SECTOR_DEFAULT_RATES`, and that per-loan array is what the simulation scales —
so a single run already carries sector-differentiated default probabilities
(the 50-loan sample portfolio produces five distinct PDs across six sectors).
Segment-targeted stress would scale that existing per-loan array; it does not
require an engine rewrite. What does not exist is a primary source for how much
harder a retail book should be shocked than a multifamily one, so the per-sector
factors would be invented. If you want this capability, scope a calibration
source, not an engine. See "Known limitations" item 1 in the README.

In [9]:
deep_stress = create_recession_scenario(
    noi_shock=-0.30,
    property_value_shock=-0.35,
    default_rate_multiplier=3.0,
    name="Deep Portfolio Stress",
    severity="severe",
)

result_deep = engine.run_simulation(deep_stress, n_iterations=500, seed=0)
print(f"Scenario: {deep_stress.name}")
print(f"Expected Loss:    ${result_deep.expected_loss:,.0f}")
print(f"VaR (95%):        ${result_deep.var_95:,.0f}")

Scenario: Deep Portfolio Stress
Expected Loss:    $6,381,469
VaR (95%):        $12,357,796


## 9. Capital Adequacy Deep Dive

Analyze whether the CDFI has enough capital to absorb stress-case losses.


In [10]:
# Use the engine's own simulated loss distribution, not a synthetic stand-in.
result = engine.run_simulation(from_standard("2008_recession"), n_iterations=1000, seed=42)
losses = engine.loss_distribution

cap_report = capital_adequacy_report(
    available_capital=5_000_000,
    losses=losses,
)
print("Capital Adequacy Report:")
print("=" * 50)
for k, v in cap_report.items():
    print(f"  {k:<20} {v:>18,.4f}")

Capital Adequacy Report:
  available_capital        5,000,000.0000
  expected_loss            8,843,296.9130
  var_95                  15,033,139.3438
  var_99                  17,396,437.8562
  cvar_99                 20,104,925.2590
  car_vs_el                        0.5654
  car_vs_var99                     0.2874
  car_vs_cvar99                    0.2487
  breaches                       858.0000
  breach_rate                      0.8580


## 10. Risk Metric Deep Dive

VaR, CVaR, and tail loss for any loss distribution.


In [11]:
var_95 = value_at_risk(losses, confidence=0.95)
var_99 = value_at_risk(losses, confidence=0.99)
cvar_95 = conditional_var(losses, confidence=0.95)
exp_loss = expected_loss(losses)
tail = tail_loss(losses, pct=0.01)   # mean of the worst 1% of paths

print("Risk Metrics from Loss Distribution:")
print("=" * 50)
print(f"  Mean Expected Loss:               ${exp_loss:,.0f}")
print(f"  VaR at 95%:                       ${var_95:,.0f}")
print(f"  VaR at 99%:                       ${var_99:,.0f}")
print(f"  CVaR at 95% (Expected Shortfall): ${cvar_95:,.0f}")
print(f"  Tail Loss (worst 1%):             ${tail:,.0f}")

Risk Metrics from Loss Distribution:
  Mean Expected Loss:               $8,843,297
  VaR at 95%:                       $15,033,139
  VaR at 99%:                       $17,396,438
  CVaR at 95% (Expected Shortfall): $16,771,524
  Tail Loss (worst 1%):             $20,104,925


## Summary

This notebook demonstrated the complete CDFI stress testing workflow:

1. **Generate realistic CDFI portfolios** — 50 loans across sectors
2. **Standard stress scenarios** — 2008 recession, COVID, rate spike, regional CRE crash
3. **Monte Carlo simulation** — 1,000 iterations
4. **Stress reports** — formatted plain-text summaries
5. **Scenario comparison** — side-by-side results across scenarios
6. **Custom scenarios** — build and label your own stress tests
7. **Capital adequacy** — does the CDFI have enough buffer?
8. **Risk metrics** — VaR, CVaR, tail loss deep dive

Before using any of these numbers in a submission, read the **Known limitations**
section of the README: shocks are portfolio-wide, only the NOI draw feeds the loss
distribution, and the calibrations are house assumptions.

Every **code** cell in this notebook is executed by CI
(`tests/test_committed_artifacts.py`) and its committed output is byte-compared against a
fresh run, so the code and the printed figures cannot drift away from the library's actual
API.

The **markdown** prose is not executed, so it gets none of that guarantee for free. Its
load-bearing claims are gated separately by `tests/test_documented_figures.py`, which sweeps
this notebook's markdown alongside the README, the CHANGELOG and the shipped source. That
gap is why the 0.2.0 sector-scenario wording was corrected in the README and the
`create_recession_scenario` docstring but left stale in this notebook until it was caught.

**GitHub:** https://github.com/Jaypatel1511/cdfi-stress-tester
**PyPI:** https://pypi.org/project/cdfi-stress-tester